In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import pandas as pd
import emoji
from transformers import pipeline


In [ ]:
import pandas as pd

sample = pd.read_csv("/content/drive/MyDrive/emoji_sample.csv")


In [ ]:

def load_data():
  return pd.read_csv("/content/drive/MyDrive/emoji_sample.csv")

In [ ]:
#sentiment shift

def reorganize_emojis(text):
    return emoji.replace_emoji(str(text), replace="")

# using "cardiffnlp/twitter-roberta-base-sentiment-latest" to utilize RoBERTa for Twitter text from HuggingFace
def sentiment():
    return pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")


def sentiment_shift(classifier, txt, maximum=256):
    test= str(txt)[:maximum]


    if test.strip()=="":
        return 0

    x = classifier(test)[0]

    actual = x["label"]
    val = x["score"]


    if actual == "positive":
        return val

    if actual == "negative":
        return val*(-1)

    else:
        return 0



In [ ]:
def sampler(df, nl=30):
    return df.sample(n=nl, random_state=7).reset_index(drop=True)


def shorten(actual, val):
  if actual == "positive":
    return val
  if actual == "negative":
    return val*(-1)
  return 0

def sentiment_analysis():
    data = load_data()
    cls = sentiment()


    sample = sampler(data, nl=200)

    withh = []
    without = []
    shift = []

    texts_with = [str(t)[:256] for t in sample["Text"]]
    texts_wout = [reorganize_emojis(t)[:256] for t in sample["Text"]]

    w = cls(texts_with, batch_size=30, truncation=True)
    wout = cls(texts_wout, batch_size=30, truncation=True)

    withh = [shorten(i["label"], i["score"]) for i in w]
    without = [shorten(i["label"], i["score"]) for i in wout]
    shift = [i-j for i,j in zip(withh, without)]

    sample["sent_with_sentiment"] = withh
    sample["sent_without_sentiment"] = without
    sample["sent_shifts"] = shift



    collected = sample.groupby("label")["sent_shifts"]
    mean = collected.mean()

    summ = mean.reset_index()
    summ.columns = ["label", "mean_shift"]
    summ = summ.sort_values("mean_shift", ascending=False)

    print(summ)




In [ ]:
sentiment_analysis()

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


                            label  mean_shift
9              face_savoring_food    0.814167
2               check_mark_button    0.676114
34       smiling_face_with_hearts    0.513980
31                   smiling_face    0.347442
33   smiling_face_with_heart-eyes    0.297171
18                       hot_face    0.259918
23                  partying_face    0.243229
16       grinning_face_with_sweat    0.234077
32         smiling_face_with_halo    0.227117
26                    rabbit_face    0.194948
42                   winking_face    0.191298
13                           fire    0.167629
10      face_with_steam_from_nose    0.166812
37                       sparkles    0.135592
36         smiling_face_with_tear    0.122863
39                  thinking_face    0.118982
38                            sun    0.114907
40                      thumbs_up    0.074091
28  rolling_on_the_floor_laughing    0.068371
27                      red_heart    0.060003
7                            eyes 